In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "perfumes.csv"
)

df_rec = df[
    (df["vote_count"] >= 50) &
    (df["rating_avg"] > 0)
].copy()

df_rec.shape

(36732, 59)

In [2]:
text_cols = [
    "description",
    "accords",
    "notes_top",
    "notes_middle",
    "notes_base",
    "notes_flat"
]

for col in text_cols:
    df_rec[col] = df_rec[col].fillna("")

df_rec["combined_text"] = (
    df_rec["description"] + " " +
    df_rec["accords"] + " " +
    df_rec["notes_top"] + " " +
    df_rec["notes_middle"] + " " +
    df_rec["notes_base"] + " " +
    df_rec["notes_flat"]
)

Clean text

In [3]:
df_rec["combined_text"] = (
    df_rec["combined_text"]
    .str.lower()
    .str.replace("|", " ", regex=False)
    .str.replace(":", " ", regex=False)
    .str.replace("/", " ", regex=False)
)

TF-IDF Vectorizer

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

tfidf_matrix = tfidf.fit_transform(
    df_rec["combined_text"]
)

tfidf_matrix.shape

(36732, 5000)

In [5]:
def search_perfume(keyword, top_n=10):
    results = df_rec[
        df_rec["name"].str.contains(
            keyword,
            case=False,
            na=False
        )
    ][
        [
            "name",
            "brand",
            "rating_avg",
            "vote_count"
        ]
    ].sort_values(
        "vote_count",
        ascending=False
    ).head(top_n)

    return results

In [6]:
search_perfume("sauvage")

,name,brand,rating_avg,vote_count
31191,Sauvage,Dior,3.8564,32551
66855,Sauvage Elixir,Dior,4.3045,18181
46959,Sauvage Eau de Parfum,Dior,4.2115,14088
227,Eau Sauvage,Dior,4.2235,6336
55007,Sauvage Parfum,Dior,4.1024,5948
93751,Sauvage Eau Forte,Dior,2.4380,2806
14551,Eau Sauvage Parfum,Dior,4.3682,2759
42936,Eau Sauvage Parfum 2017,Dior,4.4137,2654
30206,Eau Sauvage Cologne,Dior,4.1962,1376
228,Eau Sauvage Extreme,Dior,4.1817,1216


### Recommendation Function

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_perfume(perfume_index, top_n=10):
    similarity_scores = cosine_similarity(
        tfidf_matrix[perfume_index],
        tfidf_matrix
    ).flatten()

    similar_indices = similarity_scores.argsort()[::-1][1:top_n+1]

    recommendations = df_rec.iloc[similar_indices][
        [
            "name",
            "brand",
            "rating_avg",
            "vote_count",
            "gender",
            "accords"
        ]
    ].copy()

    recommendations["similarity_score"] = similarity_scores[similar_indices]

    return recommendations

In [8]:
search_perfume("sauvage")

,name,brand,rating_avg,vote_count
31191,Sauvage,Dior,3.8564,32551
66855,Sauvage Elixir,Dior,4.3045,18181
46959,Sauvage Eau de Parfum,Dior,4.2115,14088
227,Eau Sauvage,Dior,4.2235,6336
55007,Sauvage Parfum,Dior,4.1024,5948
93751,Sauvage Eau Forte,Dior,2.4380,2806
14551,Eau Sauvage Parfum,Dior,4.3682,2759
42936,Eau Sauvage Parfum 2017,Dior,4.4137,2654
30206,Eau Sauvage Cologne,Dior,4.1962,1376
228,Eau Sauvage Extreme,Dior,4.1817,1216


In [9]:
recommend_perfume(
    perfume_index=31191,
    top_n=10
)

,name,brand,rating_avg,vote_count,gender,accords,similarity_score
2313,Jubilation XXV Man,Amouage,4.3790,6691,male,amber:100|sweet:67|warm spicy:53|woody:50|bals...,0.487773
85309,Jubilant,Killer Oud,4.4000,125,unisex,amber:100|sweet:71|warm spicy:57|woody:48|bals...,0.428468
102532,Mouj Jubilee XXVI,Milestone Perfumes,3.8947,57,unisex,amber:100|warm spicy:66|woody:63|sweet:51|bals...,0.390224
46660,Golden Rush,Alexandria Fragrances,4.4103,78,male,amber:100|woody:78|warm spicy:66|sweet:63|arom...,0.387999
82191,Spectre 575 - 149 Shadows Of Bergamot,Alûstre,4.2679,56,unisex,warm spicy:100|amber:75|aromatic:48|vanilla:43...,0.310831
15021,Interlude Man,Amouage,4.1316,8823,male,amber:100|balsamic:45|fresh spicy:43|smoky:38|...,0.300754
60427,New York Intense,Fragrance Du Bois,4.1223,368,unisex,warm spicy:100|woody:77|amber:71|sweet:62|arom...,0.300127
48061,Cambridge Knight,English Laundry,4.0471,361,male,woody:100|aromatic:65|fresh spicy:53|fruity:52...,0.283303
50301,Feu Patchouli,Maison Rebatchi,4.0755,265,unisex,citrus:100|warm spicy:89|woody:78|patchouli:57...,0.268647
61201,Brave Man,La Rive,3.7171,251,male,citrus:100|marine:55|aromatic:53|woody:43|fres...,0.265274


In [13]:
def recommend_perfume(perfume_index,
                      top_n=10,
                      min_votes=500):

    # Calculate similarity
    similarity = cosine_similarity(
        tfidf_matrix[perfume_index],
        tfidf_matrix
    ).flatten()

    # Create recommendations dataframe
    recommendations = df_rec.copy()

    recommendations["similarity_score"] = similarity

    # Remove the selected perfume
    recommendations = recommendations.drop(index=perfume_index)

    # Filter low vote perfumes
    recommendations = recommendations[
        recommendations["vote_count"] >= min_votes
    ]

    # Create hybrid score
    recommendations["vote_score"] = np.log1p(
        recommendations["vote_count"]
    )

    recommendations["hybrid_score"] = (
        0.70 * recommendations["similarity_score"] +
        0.20 * (recommendations["rating_avg"] / 5) +
        0.10 * (
            recommendations["vote_score"] /
            recommendations["vote_score"].max()
        )
    )

    # Sort by hybrid score
    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    )

    return recommendations[
        [
            "name",
            "brand",
            "rating_avg",
            "vote_count",
            "similarity_score",
            "hybrid_score",
            "accords"
        ]
    ].head(top_n)

In [14]:
recommend_perfume(
    perfume_index=31191,
    top_n=10,
    min_votes=500
)

,name,brand,rating_avg,vote_count,similarity_score,hybrid_score,accords
86006,Jubilation 40 Man,Amouage,4.4254,1361,1.000000,0.945859,amber:100|woody:70|fruity:52|warm spicy:50|bal...
2313,Jubilation XXV Man,Amouage,4.3790,6691,0.487773,0.600631,amber:100|sweet:67|warm spicy:53|woody:50|bals...
15021,Interlude Man,Amouage,4.1316,8823,0.300754,0.462460,amber:100|balsamic:45|fresh spicy:43|smoky:38|...
14868,Blackberry & Bay,Jo Malone London,4.0924,6988,0.244186,0.419070,fruity:100|fresh spicy:90|aromatic:69|citrus:6...
62695,Interlude 53 Man,Amouage,4.4247,1500,0.235527,0.411627,amber:100|smoky:74|balsamic:44|woody:44|warm s...
95785,Outlands,Amouage,4.3187,2366,0.229668,0.407631,amber:100|warm spicy:66|aromatic:63|fresh spic...
54550,Overture Man,Amouage,4.2394,2490,0.221582,0.399286,warm spicy:100|amber:90|woody:63|smoky:59|fres...
16856,Enchanted Forest,The Vagabond Prince,4.1356,2950,0.223217,0.397895,fruity:100|woody:88|aromatic:83|fresh spicy:67...
18122,Invictus,Rabanne,3.7979,14787,0.216995,0.395406,citrus:100|marine:82|aromatic:77|fresh spicy:5...
1612,Man,Calvin Klein,3.6380,1417,0.256386,0.394218,fresh spicy:100|woody:73|aromatic:69|ozonic:41...


In [15]:
def recommend_perfume(perfume_name,
                      top_n=10,
                      min_votes=500):

    # Find perfume
    matches = df_rec[
        df_rec["name"].str.contains(
            perfume_name,
            case=False,
            na=False
        )
    ]

    if matches.empty:
        print("Perfume not found.")
        return

    # If multiple perfumes match, use the most popular one
    perfume = matches.sort_values(
        "vote_count",
        ascending=False
    ).iloc[0]

    perfume_index = perfume.name

    similarity = cosine_similarity(
        tfidf_matrix[perfume_index],
        tfidf_matrix
    ).flatten()

    recommendations = df_rec.copy()

    recommendations["similarity_score"] = similarity

    # Remove the queried perfume
    recommendations = recommendations.drop(index=perfume_index)

    # Filter low-vote perfumes
    recommendations = recommendations[
        recommendations["vote_count"] >= min_votes
    ]

    # Hybrid score
    recommendations["rating_norm"] = recommendations["rating_avg"] / 5

    recommendations["vote_norm"] = (
        np.log1p(recommendations["vote_count"]) /
        np.log1p(recommendations["vote_count"]).max()
    )

    recommendations["hybrid_score"] = (
        0.70 * recommendations["similarity_score"] +
        0.20 * recommendations["rating_norm"] +
        0.10 * recommendations["vote_norm"]
    )

    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    )

    print(f"Recommendations for: {perfume['name']} ({perfume['brand']})")

    return recommendations[
        [
            "name",
            "brand",
            "rating_avg",
            "vote_count",
            "similarity_score",
            "hybrid_score"
        ]
    ].head(top_n)

In [16]:
recommend_perfume("Aventus")

Recommendations for: Aventus (Creed)


,name,brand,rating_avg,vote_count,similarity_score,hybrid_score
66606,Homem Tato,Natura,4.3621,1298,0.226447,0.401389
69274,The Blend Cardamom,O Boticário,4.3123,794,0.232948,0.399263
27614,Rose Flash,Tauerville,4.2551,643,0.221709,0.387099
2683,Un Jardin Apres la Mousson,Hermès,3.9818,4174,0.201017,0.379513
63602,"Black Pepper & Amber, Neroli",Zielinski & Rozen,4.1377,1758,0.202522,0.378557
55712,Gris Charnel,BDK Parfums,4.2187,8656,0.174163,0.377148
3712,Dark Amber & Ginger Lily,Jo Malone London,4.1222,2569,0.195685,0.376768
40081,Au Coeur du Désert,Tauer Perfumes,4.4781,2579,0.174423,0.376157
101958,Babycat Raw Bourbon,Yves Saint Laurent,4.5662,1088,0.179705,0.375151
17975,Khôl de Bahreïn,Stéphane Humbert Lucas 777,4.0596,923,0.210827,0.375105
